In [1]:
import pandas as pd 
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy import sparse 

In [2]:
r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']

#ratings_base = pd.read_csv(r"D:\IT docs\python\ttcs\ub.base", sep='\t', names=r_cols, encoding='latin-1')
#ratings_test = pd.read_csv(r"D:\IT docs\python\ttcs\ub.test", sep='\t', names=r_cols, encoding='latin-1')

ratings_base = pd.read_csv('ml-100k/ub.base', sep='\t', names=r_cols, encoding='latin-1')
ratings_test = pd.read_csv('ml-100k/ub.test', sep='\t', names=r_cols, encoding='latin-1')

rate_train = ratings_base.to_numpy()
rate_test = ratings_test.to_numpy()

rate_train[:, :2] -= 1
rate_test[:, :2] -= 1

In [3]:
class MF(object):
    def __init__(self, Y_data, K, lam = 0.1, Xinit = None, Winit = None, 
            learning_rate = 0.5, max_iter = 1000, print_every = 100, user_based = 1):
        # dữ liệu gốc dạng: [user_id, item_id, rating]
        self.Y_raw_data = Y_data

        # số chiều latent factors
        self.K = K

        # tham số regularization (hệ số phạt)
        self.lam = lam

        # tốc độ học cho thuật toán Gradient Descent
        self.learning_rate = learning_rate

        # số vòng lặp tối đa , có thể cân nhắc thay thế bằng điều kiện dừng abs(loss_new - loss_old) < 1e-4
        self.max_iter = max_iter

        # in kết quả sau mỗi |print_every| vòng lặp
        self.print_every = print_every

        # lựa chọn chuẩn hoá theo user hay theo item (user_based = 1 thì theo user, ngược lại, theo item)
        self.user_based = user_based

        # số lượng user, item và rating
        # phải cộng thêm 1 vì id bắt đầu từ 0
        self.n_users = int(np.max(Y_data[:, 0])) + 1 
        self.n_items = int(np.max(Y_data[:, 1])) + 1
        self.n_ratings = Y_data.shape[0]
        
        if Xinit is None:  # nếu chưa có giá trị khởi tạo
            self.X = np.random.randn(self.n_items, K)
        else:  # hoặc sử dụng dữ liệu đã lưu trước đó
            self.X = Xinit 
        
        if Winit is None: 
            self.W = np.random.randn(K, self.n_users)
        else:  # sử dụng dữ liệu đã lưu trước đó
            self.W = Winit
            
        # dữ liệu đã được chuẩn hoá (sẽ cập nhật sau trong hàm normalize_Y)
        self.Y_data_n = self.Y_raw_data.copy()

    def normalize_Y(self):
        # lựa chọn chuẩn hóa theo user
        if self.user_based:
            user_col = 0
            item_col = 1
            n_objects = self.n_users

        # nếu muốn chuẩn hoá theo item,
        # chỉ cần đổi vị trí 2 cột đầu của dữ liệu
        else:
            user_col = 1
            item_col = 0 
            n_objects = self.n_items

        users = self.Y_raw_data[:, user_col] 

        #mảng lưu giá trị rating trung bình 
        self.mu = np.zeros((n_objects,))

        for n in range(n_objects):

            # lấy chỉ số các dòng mà user n đã đánh giá
            # vì index phải là số nguyên nên cần ép kiểu
            ids = np.where(users == n)[0].astype(np.int32)

            # lấy các item tương ứng với những rating đó
            item_ids = self.Y_data_n[ids, item_col] 

            # lấy các giá trị rating tương ứng
            ratings = self.Y_data_n[ids, 2]

            # tính trung bình
            m = np.mean(ratings) 

            if np.isnan(m):
                m = 0  # tránh trường hợp mảng rỗng gây ra giá trị NaN

            self.mu[n] = m

            # chuẩn hoá: trừ đi giá trị trung bình
            self.Y_data_n[ids, 2] = ratings - self.mu[n]

    def loss(self):
        #khởi tạo hàm mất mát 
        L = 0

        for i in range(self.n_ratings):

            # lấy user, item, rating
            n = int(self.Y_data_n[i, 0])
            m = int(self.Y_data_n[i, 1])
            rate = self.Y_data_n[i, 2]

            # rating dự đoán = x_i^T w_u
            pred = self.X[m, :].dot(self.W[:, n])

            # cộng sai số bình phương
            L += 0.5 * (rate - pred) ** 2

        # lấy trung bình
        L /= self.n_ratings

        # thêm regularization (chuẩn Frobenius)
        L += 0.5 * self.lam * (
            np.linalg.norm(self.X, 'fro') +
            np.linalg.norm(self.W, 'fro')
        )

        return L
    
    def get_items_rated_by_user(self, user_id):
        """
        Lấy tất cả item mà user user_id đã đánh giá
        và trả về các rating tương ứng
        """
    
        # tìm các dòng có user_id ở cột 0
        ids = np.where(self.Y_data_n[:,0] == user_id)[0] 
    
        # lấy item_id ở cột 1 tại các dòng tìm được
        # chuyển sang int32 vì index phải là số nguyên
        item_ids = self.Y_data_n[ids, 1].astype(np.int32)
    
        # lấy rating ở cột 2 tại các dòng đó
        ratings = self.Y_data_n[ids, 2]
    
        # trả về tuple (item_ids, ratings)
        return (item_ids, ratings)


    def get_users_who_rate_item(self, item_id):
        """
        Lấy tất cả user đã đánh giá item item_id
        và trả về rating tương ứng
        """
    
        # tìm các dòng có item_id ở cột 1
        ids = np.where(self.Y_data_n[:,1] == item_id)[0] 
    
        # lấy user_id ở cột 0 tại các dòng tìm được
        user_ids = self.Y_data_n[ids, 0].astype(np.int32)
    
        # lấy rating ở cột 2 tại các dòng đó
        ratings = self.Y_data_n[ids, 2]
    
        # trả về tuple (user_ids, ratings)
        return (user_ids, ratings)
    
    def updateX(self):
        """
        Cập nhật ma trận đặc trưng item X bằng Gradient Descent
        """
        for m in range(self.n_items):
        
            # lấy các user đã đánh giá item m và rating tương ứng
            user_ids, ratings = self.get_users_who_rate_item(m)
        
            # lấy vector đặc trưng của các user đó
            Wm = self.W[:, user_ids]
        
            # tính gradient theo X[m,:]
            # (ratings - dự đoán)
            # cộng thêm regularization
            grad_xm = -(ratings - self.X[m, :].dot(Wm)).dot(Wm.T) / self.n_ratings \
                  + self.lam * self.X[m, :]
        
            # cập nhật X[m,:]
            self.X[m, :] -= self.learning_rate * grad_xm.reshape((self.K,))
    

    def updateW(self):
        """
        Cập nhật ma trận đặc trưng user W bằng Gradient Descent
        """
        for n in range(self.n_users):
        
            # lấy các item user n đã đánh giá và rating tương ứng
            item_ids, ratings = self.get_items_rated_by_user(n)
        
            # lấy vector đặc trưng của các item đó
            Xn = self.X[item_ids, :]
        
            # tính gradient theo W[:,n]
            grad_wn = -Xn.T.dot(ratings - Xn.dot(self.W[:, n])) / self.n_ratings \
                  + self.lam * self.W[:, n]
        
            # cập nhật W[:,n]
            self.W[:, n] -= self.learning_rate * grad_wn.reshape((self.K,))

    def fit(self):
        """
        Train model bằng Gradient Descent
        """
    
        # chuẩn hóa dữ liệu rating (trừ mean)
        self.normalize_Y()
    
        # lặp max_iter lần
        for it in range(self.max_iter):
        
            # cập nhật ma trận item
            self.updateX()
        
            # cập nhật ma trận user
            self.updateW()
        
            # in kết quả sau mỗi print_every vòng
            if (it + 1) % self.print_every == 0:
                rmse_train = self.evaluate_RMSE(self.Y_raw_data)
                print('iter =', it + 1,
                    ', loss =', self.loss(),
                    ', RMSE train =', rmse_train)


    def pred(self, u, i):
        """
        Dự đoán rating của user u cho item i
        """
    
        u = int(u)
        i = int(i)
    
        # lấy bias (mean rating)
        if self.user_based:
            bias = self.mu[u]
        else:
            bias = self.mu[i]
    
        # công thức dự đoán
        pred = self.X[i, :].dot(self.W[:, u]) + bias
    
        # giới hạn kết quả trong khoảng [0,5]
        if pred < 0:
            return 0
        if pred > 5:
            return 5
    
        return pred


    def pred_for_user(self, user_id):
        """
        Dự đoán rating cho tất cả item mà user chưa đánh giá
        """
    
        # tìm các item user đã đánh giá
        ids = np.where(self.Y_data_n[:, 0] == user_id)[0]
        items_rated_by_u = self.Y_data_n[ids, 1].tolist()
    
        # tính dự đoán cho toàn bộ item
        y_pred = self.X.dot(self.W[:, user_id]) + self.mu[user_id]
    
        predicted_ratings = []
    
        # chỉ lấy những item chưa được đánh giá
        for i in range(self.n_items):
            if i not in items_rated_by_u:
                predicted_ratings.append((i, y_pred[i]))
    
        return predicted_ratings
    
    def evaluate_RMSE(self, rate_test):
        """
        Tính Root Mean Squared Error (RMSE)
        trên tập dữ liệu test
        """
    
        # số lượng mẫu test
        n_tests = rate_test.shape[0]
    
        # tổng sai số bình phương
        SE = 0
    
        # duyệt từng mẫu test
        for n in range(n_tests):
        
            # dự đoán rating cho (user, item)
            pred = self.pred(rate_test[n, 0], rate_test[n, 1])
        
            # cộng sai số bình phương
            SE += (pred - rate_test[n, 2])**2 
    
        # tính RMSE
        RMSE = np.sqrt(SE/n_tests)
    
        return RMSE

In [4]:
rs = MF(rate_train, K = 10, lam = .1, print_every = 10, 
learning_rate = 0.75, max_iter = 100, user_based = 1)
rs.fit()
# tính toán dựa trên tập dự liệu test
RMSE = rs.evaluate_RMSE(rate_test)
print('\nUser-based MF, RMSE =', RMSE)


iter = 10 , loss = 5.638273277385114 , RMSE train = 1.2042674810864478
iter = 20 , loss = 2.6345605976832984 , RMSE train = 1.0379373463714396
iter = 30 , loss = 1.341152441732719 , RMSE train = 1.0295738391054368
iter = 40 , loss = 0.7518188791837553 , RMSE train = 1.0292228768011624
iter = 50 , loss = 0.4817431814967952 , RMSE train = 1.0292128263044473
iter = 60 , loss = 0.35790139849356956 , RMSE train = 1.0292137977197022
iter = 70 , loss = 0.3011105400429007 , RMSE train = 1.0292141943612814
iter = 80 , loss = 0.27506735351733985 , RMSE train = 1.0292142980445043
iter = 90 , loss = 0.263124423540996 , RMSE train = 1.0292143232194169
iter = 100 , loss = 0.2576476082428327 , RMSE train = 1.0292143291830733

User-based MF, RMSE = 1.0603799042551252
